In [3]:
"""
Chargement d'une instance TSPTW-D en dictionnaire
==================================================
Structure retournée — prête pour l'optimisation :

instance = {
    # --- méta ---
    "n"      : int,           # nombre de clients (hors dépôt)
    "scale"  : float,
    "horizon": float,

    # --- nœuds (index 0 = dépôt, 1..n = clients) ---
    "coords"  : {i: (x, y)},  # coordonnées
    "a"       : {i: float},    # ouverture de fenêtre
    "b"       : {i: float},    # fermeture de fenêtre
    "service" : {i: float},    # temps de service

    # --- coûts de base (distance euclidienne × scale) ---
    "c_base"  : {(i,j): float},  # symétrique, i ≠ j

    # --- perturbations ---
    "perturbations": [
        {"arc": (i,j), "t_start": float, "t_end": float, "alpha": float},
        ...
    ],
}

Fonctions exposées
------------------
charger_instance(chemin)        → dict  (chargement complet)
c_ij(instance, i, j, t)        → float (coût dynamique à l'instant t)
"""

import json
import math
from pathlib import Path


# ---------------------------------------------------------------------------
# Chargement principal
# ---------------------------------------------------------------------------

def charger_instance(chemin: str | Path) -> dict:
    """
    Charge un fichier JSON généré par `generer_dataset_tsptwd`
    et retourne un dictionnaire prêt pour l'optimisation.

    Paramètre
    ---------
    chemin : chemin vers le fichier .json

    Retourne
    --------
    instance : dict avec les clés décrites en en-tête de module.
    """
    with open(chemin, encoding="utf-8") as f:
        data = json.load(f)

    n       = data["meta"]["n_clients"]
    scale   = data["meta"]["scale"]
    horizon = data["meta"]["horizon"]

    # --- nœuds : dépôt (0) + clients (1..n) --------------------------------
    coords  = {}
    a       = {}
    b       = {}
    service = {}

    d = data["depot"]
    coords[0]  = (d["x"], d["y"])
    a[0]       = d["a"]
    b[0]       = d["b"]
    service[0] = d["service"]

    for c in data["clients"]:
        i          = c["id"]          # 1..n
        coords[i]  = (c["x"], c["y"])
        a[i]       = c["a"]
        b[i]       = c["b"]
        service[i] = c["service"]

    # --- coûts de base : distance euclidienne × scale -----------------------
    noeuds = list(range(n + 1))
    c_base = {
        (i, j): round(
            math.hypot(coords[i][0] - coords[j][0],
                       coords[i][1] - coords[j][1]) * scale,
            4,
        )
        for i in noeuds
        for j in noeuds
        if i != j
    }

    # --- perturbations ------------------------------------------------------
    perturbations = [
        {
            "arc":     tuple(p["arc"]),
            "t_start": p["t_start"],
            "t_end":   p["t_end"],
            "alpha":   p["alpha"],
        }
        for p in data.get("perturbations", [])
    ]

    instance = {
        "n":             n,
        "scale":         scale,
        "horizon":       horizon,
        "coords":        coords,
        "a":             a,
        "b":             b,
        "service":       service,
        "c_base":        c_base,
        "perturbations": perturbations,
    }

    return instance


# ---------------------------------------------------------------------------
# Coût dynamique (modèle TSPTW-D du Livrable 1)
# ---------------------------------------------------------------------------

def c_ij(instance: dict, i: int, j: int, t: float) -> float:
    """
    Coût de transit dynamique de i vers j en partant à l'instant t.

        c_ij(t) = c_base[i,j] × (1 + δ_ij(t))

    δ_ij(t) = alpha - 1  si une perturbation sur (i,j) ou (j,i) est active à t
    δ_ij(t) = 0          sinon

    Paramètres
    ----------
    instance : dict retourné par charger_instance()
    i, j     : indices des nœuds (0 = dépôt)
    t        : instant de départ (minutes)

    Retourne
    --------
    Coût en minutes (float > 0).
    """
    base = instance["c_base"][(i, j)]
    for p in instance["perturbations"]:
        arc = p["arc"]
        if arc == (i, j) or arc == (j, i):   # symétrique
            if p["t_start"] <= t <= p["t_end"]:
                return base * p["alpha"]
    return base


# ---------------------------------------------------------------------------
# Propagation temporelle (utilitaire pour les algorithmes)
# ---------------------------------------------------------------------------

def evaluer_tournee(instance: dict, permutation: list[int],
                    dynamique: bool = True) -> float:
    """
    Évalue une permutation de clients et retourne le temps total (float).

    Ignorer les fenêtres temporelles et les attentes.
    Conserve les temps de service.
    """
    service = instance["service"]
    c_base  = instance["c_base"]

    route = [0] + list(permutation) + [0]
    t     = 0.0

    for k in range(len(route) - 1):
        i, j = route[k], route[k + 1]
        t += service[i]
        transit = c_ij(instance, i, j, t) if dynamique else c_base[(i, j)]
        t += transit

    return round(t, 4)


# ---------------------------------------------------------------------------
# Affichage récapitulatif
# ---------------------------------------------------------------------------

def afficher_instance(instance: dict) -> None:
    """Affiche un résumé lisible de l'instance."""
    n = instance["n"]
    print(f"Instance TSPTW-D")
    print(f"  Clients       : {n}")
    print(f"  Horizon       : {instance['horizon']} min")
    print(f"  Scale         : {instance['scale']}")
    print(f"  Perturbations : {len(instance['perturbations'])}")
    print()
    print(f"  {'Nœud':<10} {'Coords':>20}  {'[a, b]':>20}  {'service':>8}")
    print("  " + "-" * 65)
    for i in range(n + 1):
        nom  = "Dépôt" if i == 0 else f"Client {i}"
        x, y = instance["coords"][i]
        ai, bi = instance["a"][i], instance["b"][i]
        si = instance["service"][i]
        print(f"  {nom:<10} ({x:7.3f}, {y:7.3f})  [{ai:6.1f}, {bi:6.1f}]  {si:7.1f} min")

    if instance["perturbations"]:
        print()
        print(f"  {'Arc':<10} {'t_start':>10} {'t_end':>10} {'alpha':>8}")
        print("  " + "-" * 45)
        for p in instance["perturbations"]:
            print(f"  {str(p['arc']):<10} {p['t_start']:>10.1f} {p['t_end']:>10.1f} {p['alpha']:>8.2f}")


%matplotlib inline

"""
=============================================================================
Algorithme hybride GLOP pour le TSPTW-D  —  avec visualisation temps réel
=============================================================================

Problème : Travelling Salesman Problem with Time Windows and Dynamic disruptions
           (voir livrable_1_modelisation.ipynb)

Contraintes du modèle :
  - Graphe complet, coûts symétriques c_ij(t) = c_ij_base * (1 + delta_ij(t))
  - Fenêtres temporelles [a_i, b_i] avec attente autorisée
  - Temps de service s_i
  - Perturbations : liste de dicts { 'arc':(i,j), 't_debut':float,
                                      't_fin':float,  'alpha':float }

Variable de décision : permutation sigma = (0, sigma_1, ..., sigma_n, 0)
Fonction objectif    : minimiser tau_0^retour (temps de retour au dépôt)

=============================================================================
Architecture GLOP
=============================================================================

G — Algorithme Génétique  (population)
    Initialisation greedy + aléatoire, sélection par tournoi,
    croisement OX, mutation swap / 2-opt local.

L — Local Search           (amélioration)
    2-opt et Or-opt avec évaluation dynamique des coûts c_ij(t).

O — Optimization           (renforcement élite)
    DP sur fenêtres glissantes de taille dp_window sur l'élite.

P — Perturbation           (diversification)
    Double-Bridge (4-opt non-séquentiel) sur individus stagnants.

Visualisation : 3 panneaux matplotlib mis à jour en temps réel
    - Carte de la meilleure tournée courante
    - Courbe de convergence (meilleur / moyen / pire)
    - Diversité de la population (boxplot + strip)

=============================================================================
Exemple d'utilisation
=============================================================================

    from glop_tsptwd import charger_villes_depuis_split, lancer_visualisation

    villes = charger_villes_depuis_split(chunk_size=200, source="tsp")
    lancer_visualisation(
        villes,
        perturbations   = [],
        taille_pop      = 80,
        n_generations   = 20000,
        taux_croisement = 0.85,
        taux_mutation   = 0.12,
        taux_elitisme   = 0.10,
        patience        = 350,
    )
=============================================================================
"""

# ── Imports ──────────────────────────────────────────────────────────────────
import csv
import math
import os
import random
import itertools
import time
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import matplotlib, os as _os

# Sélection automatique du backend interactif disponible.
# Priorité : TkAgg → Qt5Agg → wxAgg → Agg (headless/CI).
# Forçage possible via : MPLBACKEND=Qt5Agg python glop_tsptwd.py
if not _os.environ.get("MPLBACKEND"):
    for _backend in ("TkAgg", "Qt5Agg", "wxAgg", "Agg"):
        try:
            matplotlib.use(_backend)
            break
        except Exception:
            continue

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D



# =============================================================================
# STRUCTURES DE DONNÉES
# =============================================================================

class Ville:
    """Nœud du graphe TSPTW-D (dépôt = index 0, clients = index 1..n)."""

    def __init__(
        self,
        nom: str,
        x: float,
        y: float,
        fenetre: Tuple[float, float] = (0, float("inf")),
        service: float = 0.0,
    ):
        self.nom     = nom
        self.x       = x
        self.y       = y
        self.a       = fenetre[0]   # heure d'ouverture
        self.b       = fenetre[1]   # heure de fermeture (deadline)
        self.service = service      # temps de service s_i (minutes)

    def __repr__(self):
        return self.nom



import math
import random
import itertools
from typing import List, Tuple, Optional


# =============================================================================
# ÉVALUATION — retourne TOUJOURS un float fini
# =============================================================================

def fitness(instance: dict, permutation: List[int]) -> float:
    return evaluer_tournee(instance, permutation, dynamique=True)


# =============================================================================
# OPÉRATEURS GÉNÉTIQUES (portés depuis le notebook)
# =============================================================================

def croisement_ox(parent1: List[int], parent2: List[int]) -> List[int]:
    """Order Crossover — préserve l'ordre relatif des gènes."""
    n    = len(parent1)
    a, b = sorted(random.sample(range(n), 2))
    enfant: List[Optional[int]] = [None] * n
    enfant[a:b + 1] = parent1[a:b + 1]
    segment = set(parent1[a:b + 1])
    reste   = [g for g in parent2 if g not in segment]
    pos     = [(b + 1 + k) % n for k in range(n - (b - a + 1))]
    for i, gene in zip(pos, reste):
        enfant[i] = gene
    return enfant  # type: ignore


def mutation_swap(chromosone: List[int]) -> List[int]:
    """Échange deux positions aléatoires."""
    fils   = chromosone[:]
    i, j   = random.sample(range(len(fils)), 2)
    fils[i], fils[j] = fils[j], fils[i]
    return fils


def mutation_2opt(instance: dict, chromosone: List[int], max_iter: int = 50) -> List[int]:
    """2-opt stochastique — accepte si améliore le score pénalisé."""
    best   = chromosone[:]
    best_f = fitness(instance, best)
    n      = len(best)
    for _ in range(max_iter):
        i, j      = sorted(random.sample(range(n), 2))
        candidat  = best[:i] + best[i:j + 1][::-1] + best[j + 1:]
        f         = fitness(instance, candidat)
        if f < best_f - 1e-9:
            best, best_f = candidat, f
    return best


def or_opt(instance: dict, chromosone: List[int], max_iter: int = 40) -> List[int]:
    """Or-opt stochastique — déplace un segment de 1, 2 ou 3 clients."""
    best   = chromosone[:]
    best_f = fitness(instance, best)
    n      = len(best)
    for _ in range(max_iter):
        seg   = random.choice((1, 2, 3))
        if seg >= n:
            continue
        start   = random.randint(0, n - seg)
        segment = best[start:start + seg]
        reste   = best[:start] + best[start + seg:]
        ins     = random.randint(0, len(reste))
        candidat = reste[:ins] + segment + reste[ins:]
        f        = fitness(instance, candidat)
        if f < best_f - 1e-9:
            best, best_f = candidat, f
    return best


def double_bridge(chromosone: List[int]) -> List[int]:
    """
    Perturbation 4-opt non-séquentielle.
    Découpe A|B|C|D → recompose A|C|B|D.
    Échappe aux bassins d'attraction du 2-opt.
    """
    n = len(chromosone)
    if n < 4:
        fils = chromosone[:]
        random.shuffle(fils)
        return fils
    a, b, c = sorted(random.sample(range(1, n), 3))
    return (
        chromosone[:a]
        + chromosone[b:c]
        + chromosone[a:b]
        + chromosone[c:]
    )


def dp_reoptimise(instance: dict, chromosone: List[int], window: int = 4) -> List[int]:
    """
    Réoptimisation par énumération sur des fenêtres glissantes de taille window.
    Coût : O(n × window!) — window=4 → 24 évaluations par position.
    """
    best   = chromosone[:]
    best_f = fitness(instance, best)
    n      = len(best)
    for start in range(n - window + 1):
        sub    = best[start:start + window]
        prefix = best[:start]
        suffix = best[start + window:]
        for perm in itertools.permutations(sub):
            candidat = prefix + list(perm) + suffix
            f        = fitness(instance, candidat)
            if f < best_f - 1e-9:
                best_f = f
                best   = candidat
    return best


def selectionner_tournoi(
    population: List[List[int]],
    scores: List[float],
    k: int = 3,
) -> List[int]:
    """Sélection par tournoi de taille k."""
    indices = random.sample(range(len(population)), k)
    gagnant = min(indices, key=lambda i: scores[i])
    return population[gagnant][:]


# =============================================================================
# INITIALISATION DE LA POPULATION
# =============================================================================

def initialiser_population(instance: dict, taille_pop: int) -> List[List[int]]:
    """
    - 1 individu greedy (nearest-neighbor depuis le dépôt)
    - 1 individu trié par deadline (EDF)
    - reste : permutations aléatoires
    """
    n       = instance["n"]
    clients = list(range(1, n + 1))
    pop     = []

    # Greedy nearest-neighbor
    non_visites = clients[:]
    current     = 0
    greedy      = []
    while non_visites:
        suivant = min(non_visites, key=lambda j: instance["c_base"][(current, j)])
        greedy.append(suivant)
        non_visites.remove(suivant)
        current = suivant
    pop.append(greedy)

    # EDF (Earliest Deadline First)
    pop.append(sorted(clients, key=lambda i: instance["b"][i]))

    # Aléatoires
    for _ in range(taille_pop - 2):
        ind = clients[:]
        random.shuffle(ind)
        pop.append(ind)

    return pop


# =============================================================================
# ALGORITHME GLOP PRINCIPAL
# =============================================================================

def glop_tsptwd(
    instance: dict,
    taille_pop: int = 60,
    n_generations: int = 2000,
    taux_croisement: float = 0.85,
    taux_mutation: float = 0.15,
    taux_elitisme: float = 0.10,
    patience: int = 300,
    dp_window: int = 4,
    seed: int = None,
) -> dict:
    """
    GLOP pour TD-TSP (Time-Dependent TSP).

    Architecture (G-L-O-P) :
    ─────────────────────────────────────────────────────────────────
    G — Génétique  : population, sélection tournoi, croisement OX
    L — Local      : mutation 2-opt + or-opt avec score dynamique
    O — Optimisation : DP fenêtre glissante sur l'élite
    P — Perturbation : double-bridge si stagnation
    ─────────────────────────────────────────────────────────────────
    """
    if seed is not None:
        random.seed(seed)

    n           = instance["n"]
    elite_size  = max(1, int(taille_pop * taux_elitisme))
    historique  = []           # meilleur score par génération

    # ── Initialisation ────────────────────────────────────────────────────────
    population = initialiser_population(instance, taille_pop)
    scores     = [fitness(instance, ind) for ind in population]

    meilleur_global      = min(population, key=lambda x: fitness(instance, x))[:]
    meilleur_score_global = fitness(instance, meilleur_global)
    sans_amelio           = 0

    for gen in range(n_generations):

        # ── Tri + élite ───────────────────────────────────────────────────────
        ordre      = sorted(range(len(population)), key=lambda i: scores[i])
        population = [population[i] for i in ordre]
        scores     = [scores[i]     for i in ordre]

        historique.append(scores[0])

        if scores[0] < meilleur_score_global - 1e-9:
            meilleur_score_global = scores[0]
            meilleur_global       = population[0][:]
            sans_amelio           = 0
        else:
            sans_amelio += 1

        # ── Phase O : DP sur l'élite ──────────────────────────────────────────
        if gen % max(1, patience // 5) == 0:
            for i in range(elite_size):
                population[i] = dp_reoptimise(instance, population[i], dp_window)
                scores[i]     = fitness(instance, population[i])

        # ── Phase P : double-bridge si stagnation ─────────────────────────────
        if sans_amelio > patience // 3:
            for i in range(elite_size, min(elite_size + 3, taille_pop)):
                population[i] = double_bridge(population[i])
                scores[i]     = fitness(instance, population[i])

        # ── Arrêt anticipé ────────────────────────────────────────────────────
        if sans_amelio >= patience:
            break

        # ── Nouvelle génération ───────────────────────────────────────────────
        nouvelle_pop = population[:elite_size]   # élite conservée

        while len(nouvelle_pop) < taille_pop:
            # Croisement
            if random.random() < taux_croisement:
                p1     = selectionner_tournoi(population, scores)
                p2     = selectionner_tournoi(population, scores)
                enfant = croisement_ox(p1, p2)
            else:
                enfant = selectionner_tournoi(population, scores)

            # Mutation
            r = random.random()
            if r < taux_mutation:
                enfant = mutation_swap(enfant)
            elif r < taux_mutation * 2:
                enfant = mutation_2opt(instance, enfant, max_iter=30)

            # Or-opt avec proba 0.30
            if random.random() < 0.30:
                enfant = or_opt(instance, enfant, max_iter=20)

            nouvelle_pop.append(enfant)

        population = nouvelle_pop
        scores     = [fitness(instance, ind) for ind in population]

    cout_final = evaluer_tournee(instance, meilleur_global, dynamique=True)

    return {
        "meilleure_tournee": meilleur_global,
        "cout": cout_final,
        "realisable": True,
        "historique": historique,
    }

CHEMIN_JSON = "datasets/tsptwd_n200.json"

instance = charger_instance(CHEMIN_JSON)

resultat = glop_tsptwd(
    instance,
    taille_pop      = 60,
    n_generations   = 2000,
    taux_croisement = 0.85,
    taux_mutation   = 0.15,
    taux_elitisme   = 0.10,
    patience        = 300,
    dp_window       = 4,
    seed            = 42,
)

print(f"Réalisable : {resultat['realisable']}")
print(f"Coût total : {resultat['cout']:.4f}")
print(f"Générations: {len(resultat['historique'])}")
print(f"Tournée    : {resultat['meilleure_tournee']}")

Réalisable : True
Coût total : 4226.4825
Générations: 1108
Tournée    : [130, 72, 135, 119, 50, 36, 98, 139, 163, 64, 117, 30, 66, 166, 171, 10, 70, 21, 67, 102, 12, 16, 138, 197, 149, 44, 158, 99, 69, 32, 185, 95, 53, 155, 114, 104, 150, 15, 190, 124, 151, 84, 131, 28, 56, 13, 110, 133, 75, 62, 140, 4, 118, 161, 181, 186, 160, 193, 89, 17, 143, 65, 189, 38, 179, 71, 97, 198, 73, 51, 123, 34, 49, 43, 2, 52, 194, 27, 54, 159, 183, 142, 55, 195, 187, 68, 88, 29, 122, 170, 113, 191, 5, 184, 126, 23, 154, 11, 106, 157, 137, 172, 111, 41, 1, 63, 178, 19, 92, 35, 162, 188, 129, 61, 167, 6, 128, 90, 101, 9, 174, 45, 145, 173, 168, 141, 132, 152, 147, 37, 103, 182, 20, 85, 116, 58, 26, 200, 144, 93, 91, 169, 176, 120, 46, 59, 177, 47, 107, 24, 112, 115, 125, 18, 48, 80, 40, 94, 78, 153, 148, 31, 180, 22, 196, 3, 105, 74, 42, 39, 108, 165, 33, 7, 192, 199, 14, 82, 121, 83, 25, 57, 146, 81, 96, 77, 76, 109, 134, 127, 164, 156, 86, 100, 175, 136, 87, 79, 8, 60]
